In [18]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer

In [19]:
seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
s = System([Protein(rep=seq, id='EcCM', first_index=2)])
inst = s.rep_to_instance()

In [ ]:
m = BoltzFoldTransformer(device='cpu', use_msa_server=True, diffusion_samples=1)
m.build(s)

In [21]:
output_structures = m.transform([inst])

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_p2x3md51/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:04<00:00,  4.94s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-15 10:42:59.535 | INFO     | evedesign.models.boltzfold:_load_model:206 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-15 10:53:26.300 | INFO     | evedesign.models.boltzfold:transform:361 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_p2x3md51/predictions
2026-04-15 10:53:26.314 | INFO     | evedesign.models.boltzfold:transform:362 - Files written (16):
2026-04-15 10:53:26.317 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_0.json (443 bytes)
2026-04-15 10:53:26.318 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_mod

In [8]:
result = output_structures[0]
print(f"Score (ptm): {result.score}")
print(f"Confidence (pLDDT): {result.confidence}")

Score (ptm): 0.8622773289680481
Confidence (pLDDT): 0.9394526481628418


In [9]:
print("Confidence scores:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")

# Structure from EntityInstance.models
ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]
    print(f"\nChain: {chain_id}")
    print(f"Atom count: {len(structure.atom_array)}")
    print(f"Residue range: {structure.atom_array.res_id.min()} - {structure.atom_array.res_id.max()}")
    print(structure.atom_df().head(5))

Confidence scores:
  confidence_score: 0.924017608165741
  ptm: 0.8622773289680481
  iptm: 0.0
  ligand_iptm: 0.0
  protein_iptm: 0.0
  complex_plddt: 0.9394526481628418
  complex_iplddt: 0.9394526481628418
  complex_pde: 0.3468779921531677
  complex_ipde: 0.0
  chains_ptm: {'0': 0.8622773289680481}
  pair_chains_iptm: {'0': {'0': 0.8622773289680481}}

Chain: A
Atom count: 762
Residue range: 2 - 95
  chain_id  res_id ins_code res_name  hetero atom_name element  atom_id  \
0        A       2               THR   False         N       N        1   
1        A       2               THR   False        CA       C        2   
2        A       2               THR   False         C       C        3   
3        A       2               THR   False         O       O        4   
4        A       2               THR   False        CB       C        5   

   b_factor  occupancy  charge        x          y         z  
0    62.931        1.0       0  4.90606 -30.878660 -14.77777  
1    62.931        1.

In [10]:
! pip install py3Dmol -q

In [11]:
import io
import py3Dmol

ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]

    buf = io.StringIO()
    structure.to_file(buf, format="cif")
    cif_content = buf.getvalue()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_content, "cif")
    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90
            }
        }
    })
    view.zoomTo()
    view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Test (a): single protein, single instance

In [ ]:
s_a = System([Protein(rep='MAST', id='test_a')])
results_a = BoltzFoldTransformer(
    device='cpu', use_msa=False,
    sampling_steps=1, recycling_steps=1
).build(s_a).transform([s_a.rep_to_instance()])
ei = results_a[0][0]
assert ei.models is not None, "models is None"
assert results_a[0].score is not None, "score is None"

### Test (b): multiple instances

In [ ]:
s_b = System([Protein(rep='MAST', id='test_b')])
inst_b1 = s_b.rep_to_instance()
inst_b2 = s_b.rep_to_instance()
results_b = BoltzFoldTransformer(
    device='cpu', use_msa_server=True,
).build(s_b).transform([inst_b1, inst_b2])
assert len(results_b) == 2, f"expected 2 results got {len(results_b)}"
assert results_b[0][0].models is not None
assert results_b[1][0].models is not None


### Test (c): two-chain complex

In [16]:
s_c = System([
    Protein(rep='MAST', id='chain1'),
    Protein(rep='GKLT', id='chain2'),
])
results_c = BoltzFoldTransformer(
    device='cpu', use_msa_server=True,
).build(s_c).transform([s_c.rep_to_instance()])
ei0 = results_c[0][0]
ei1 = results_c[0][1]
assert ei0.models is not None, "entity 0 models is None"
assert ei1.models is not None, "entity 1 models is None"
assert list(ei0.models.keys()) == ["A"], \
    f"expected chain A got {list(ei0.models.keys())}"
assert list(ei1.models.keys()) == ["B"], \
    f"expected chain B got {list(ei1.models.keys())}"

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_d3httwdk/inputs/instance_0.yaml with 2 protein entities.
Calling MSA server for target instance_0 with 2 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


Sleeping for 6s. Reason: PENDING
COMPLETE: 100%|██████████| 300/300 [elapsed: 00:08 remaining: 00:00]
Sleeping for 5s. Reason: PENDING
100%|██████████| 1/1 [00:15<00:00, 15.34s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-13 17:06:38.601 | INFO     | evedesign.models.boltzfold:_load_model:206 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-13 17:08:27.335 | INFO     | evedesign.models.boltzfold:transform:361 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_d3httwdk/predictions
2026-04-13 17:08:27.345 | INFO     | evedesign.models.boltzfold:transform:362 - Files written (26):
2026-04-13 17:08:27.348 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_0.

In [17]:
results_c

[SystemInstance([EntityInstance(rep=MAST, models=1), EntityInstance(rep=GKLT, models=1)] id=None score=0.1751897633075714)]

### Test (d): entity parameter

In [ ]:
s_d = System([Protein(rep='MAST', id='test_d')])
m_d = BoltzFoldTransformer(
    device='cpu', use_msa=False,
    sampling_steps=1, recycling_steps=1
).build(s_d)
try:
    m_d.transform([s_d.rep_to_instance()], entity=1)
    print("FAIL: expected NotImplementedError")
except NotImplementedError:
    print("PASS: NotImplementedError raised as expected")

###  Test (e): Multiple diffusion_samples

In [ ]:
s_e = System([Protein(rep='MAST', id='test_e')])
results_e = BoltzFoldTransformer(
    device='cpu', use_msa=False,
    sampling_steps=1, recycling_steps=1,
    diffusion_samples=3,
).build(s_e).transform([s_e.rep_to_instance()])
ei = results_e[0][0]
assert ei.models is not None, "models is None"
assert results_e[0].score is not None, "score is None"
print(f"PASS: got structure from best of 3 samples")
print(f"  score: {results_e[0].score}")

### Test (f): homo-oligomer copies=2

In [27]:
s_f = System([Protein(rep='MAST', id='test_f', copies=2)])
results_f = BoltzFoldTransformer(
    device='cpu', use_msa=False,
).build(s_f).transform([s_f.rep_to_instance()])
ei = results_f[0][0]
assert ei.models is not None, "models is None"
chains = list(ei.models.keys())
assert "A" in chains, f"chain A missing, got {chains}"
assert "B" in chains, f"chain B missing, got {chains}"
print(f"PASS: both chains present: {chains}")


Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Found explicit empty MSA for some proteins, will run these in single sequence mode. Keep in mind that the model predictions will be suboptimal without an MSA.


100%|██████████| 1/1 [00:00<00:00, 13.29it/s]
2026-04-15 11:00:17.362 | INFO     | evedesign.models.boltzfold:_load_model:206 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-15 11:02:29.016 | INFO     | evedesign.models.boltzfold:transform:361 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold__gdekj84/predictions
2026-04-15 11:02:29.028 | INFO     | evedesign.models.boltzfold:transform:362 - Files written (26):
2026-04-15 11:02:29.030 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_0.json (672 bytes)
2026-04-15 11:02:29.031 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_1.json (673 bytes)
2026-04-15 11:02:29.032 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_2.json (674 bytes)
2026-04-15 11:02:29.033 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence

PASS: both chains present: ['A', 'B']


### Test (g): entity with MSA sequences

In [24]:
from evedesign.sequence import Sequences, Sequence
seqs = Sequences([
    Sequence(seq='MAST', id='hom1'),
    Sequence(seq='VAST', id='hom2'),
])
s_g = System([
    Protein(rep='MAST', id='test_g', sequences=seqs)
])
results_g = BoltzFoldTransformer(
    device='cpu', use_msa=True,
).build(s_g).transform([s_g.rep_to_instance()])
ei = results_g[0][0]
assert ei.models is not None, "models is None"


Processing 1 inputs with 1 threads.


100%|██████████| 1/1 [00:00<00:00,  2.67it/s]
2026-04-15 10:56:10.960 | INFO     | evedesign.models.boltzfold:_load_model:206 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-15 10:57:17.814 | INFO     | evedesign.models.boltzfold:transform:361 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_1nk75e5i/predictions
2026-04-15 10:57:17.820 | INFO     | evedesign.models.boltzfold:transform:362 - Files written (26):
2026-04-15 10:57:17.822 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_0.json (444 bytes)
2026-04-15 10:57:17.823 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_1.json (444 bytes)
2026-04-15 10:57:17.824 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence_instance_0_model_2.json (443 bytes)
2026-04-15 10:57:17.825 | INFO     | evedesign.models.boltzfold:transform:365 -   instance_0/confidence

### Test (i): non-protein entity

In [22]:
from evedesign.system import DNA
s_i = System([DNA(rep='ATCG', id='test_i')])
m_i = BoltzFoldTransformer(device='mps', use_msa=False)
ok, msg = m_i.can_model(s_i)
assert not ok, "can_model should return False for DNA"
try:
    m_i.build(s_i)
    print("FAIL: expected ValueError from build()")
except ValueError as e:
    print(f"PASS: correctly rejected with: {e}")

PASS: correctly rejected with: Entity 0 has type 'dna'. Only protein entities are supported. DNA/RNA/ligand coming soon.
